# 02 — RVC v2 Fine-Tuning
**Neural Voice Identity Control & Deepfake Audio Analysis — Disertație 2026**

Antrenează modele RVC v2 pentru cele 3 voci. Rulează celulele **în ordine, de sus în jos**.

**Prerequisit:** `01_data_collection.ipynb` rulat și clipuri existente pe Drive.

**Timp estimat:** ~1-2h per voce pe T4 GPU (~4-6h total pentru 3 voci).

---
> ⚡ **Runtime → Change runtime type → T4 GPU** înainte să începi.

In [ ]:
# Celula 1 — Verificare GPU
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('EROARE: Nu s-a detectat GPU.')
    print('Mergi la: Runtime → Change runtime type → GPU → T4')

In [ ]:
# Celula 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('Drive montat.')

In [ ]:
# Celula 3 — Clone RVC
import os

RVC_DIR = '/content/RVC'

if not os.path.exists(RVC_DIR):
    print('Clonare RVC...')
    !git clone https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git {RVC_DIR}
else:
    print('RVC deja exista, actualizez...')
    !git -C {RVC_DIR} pull

os.chdir(RVC_DIR)
print(f'OK. Folder: {os.getcwd()}')

In [ ]:
# Celula 4 — Instalare pachete
import subprocess, sys

def pip(pkg, extra=[]):
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg] + extra,
                       capture_output=True, text=True)
    print(f'  [{"OK" if r.returncode == 0 else "FAIL"}] {pkg}')
    return r.returncode == 0

print('Instalare...')
for pkg in [
    'numpy', 'scipy', 'librosa==0.9.2', 'soundfile',
    'praat-parselmouth>=0.4.2', 'torchcrepe',
    'faiss-cpu', 'ffmpeg-python', 'av',
    'tensorboard', 'transformers', 'huggingface_hub',
    'gradio', 'numba', 'einops', 'local_attention',
]:
    pip(pkg)

if not pip('pyworld'):
    pip('pyworld', ['--no-build-isolation'])

print('Gata!')

In [ ]:
# Celula 5 — Download modele pretrained RVC v2
import os

os.makedirs(f'{RVC_DIR}/assets/pretrained_v2', exist_ok=True)
os.makedirs(f'{RVC_DIR}/assets/hubert', exist_ok=True)
os.makedirs(f'{RVC_DIR}/assets/rmvpe', exist_ok=True)

BASE = 'https://huggingface.co/lj1995/VoiceConversionWebUI/resolve/main'
files = [
    (f'{BASE}/pretrained_v2/f0G40k.pth', f'{RVC_DIR}/assets/pretrained_v2/f0G40k.pth'),
    (f'{BASE}/pretrained_v2/f0D40k.pth', f'{RVC_DIR}/assets/pretrained_v2/f0D40k.pth'),
    (f'{BASE}/pretrained_v2/f0G48k.pth', f'{RVC_DIR}/assets/pretrained_v2/f0G48k.pth'),
    (f'{BASE}/pretrained_v2/f0D48k.pth', f'{RVC_DIR}/assets/pretrained_v2/f0D48k.pth'),
    (f'{BASE}/hubert_base.pt',            f'{RVC_DIR}/assets/hubert/hubert_base.pt'),
    (f'{BASE}/rmvpe.pt',                  f'{RVC_DIR}/assets/rmvpe/rmvpe.pt'),
]

for url, dest in files:
    if not os.path.exists(dest):
        print(f'Download: {os.path.basename(dest)}')
        !wget -q '{url}' -O '{dest}'
    else:
        print(f'  OK (exista): {os.path.basename(dest)}')

print('Modele gata.')

## Configurare voci

Celula de mai jos defineste cele 3 voci de antrenat. Nu modifica nimic daca vrei sa antrenezi toate 3.

In [ ]:
# Celula 6 — Configurare
import os

DRIVE_BASE = '/content/drive/MyDrive/disertatie'
CLIPS_BASE = f'{DRIVE_BASE}/data/processed'
MODELS_OUT = f'{DRIVE_BASE}/models'
os.makedirs(MODELS_OUT, exist_ok=True)

VOICES = [
    {'voice_id': 'voice_1', 'name': 'Calin Georgescu',  'sr': 40000, 'epochs': 200, 'batch': 8, 'save_every': 50},
    {'voice_id': 'voice_2', 'name': 'Nicusor Dan',      'sr': 40000, 'epochs': 200, 'batch': 8, 'save_every': 50},
    {'voice_id': 'voice_3', 'name': 'Diana Sosoaca',    'sr': 40000, 'epochs': 200, 'batch': 8, 'save_every': 50},
]

print('Configurare:')
for v in VOICES:
    clips = f"{CLIPS_BASE}/{v['voice_id']}/clips"
    n = len([f for f in os.listdir(clips) if f.endswith('.wav')]) if os.path.exists(clips) else 0
    print(f"  {v['name']}: {n} clipuri")

In [ ]:
# Celula 7 — Functii de antrenare (toate fix-urile incluse)
import subprocess, shutil, sys, threading, time, glob, os, json
import numpy as np
import soundfile as sf
import torch
sys.path.insert(0, RVC_DIR)


# ── Fix 1: config.json cu parametri corecti pentru f0G40k.pth ──────────────
def _create_config(log_dir, sr, batch_size):
    config = {
        "train": {
            "log_interval": 200, "seed": 1234, "epochs": 20000,
            "learning_rate": 1e-4, "betas": [0.8, 0.99], "eps": 1e-9,
            "batch_size": batch_size, "fp16_run": True, "lr_decay": 0.999875,
            "segment_size": 12800, "init_lr_ratio": 1, "warmup_epochs": 0,
            "c_mel": 45, "c_kl": 1.0
        },
        "data": {
            "max_wav_value": 32768.0, "sampling_rate": sr,
            "filter_length": 2048, "hop_length": sr // 100,
            "win_length": 2048, "n_mel_channels": 125,
            "mel_fmin": 0.0, "mel_fmax": None
        },
        "model": {
            "inter_channels": 192, "hidden_channels": 192,
            "filter_channels": 768, "n_heads": 2, "n_layers": 6,
            "kernel_size": 3, "p_dropout": 0, "resblock": "1",
            "resblock_kernel_sizes": [3, 7, 11],
            "resblock_dilation_sizes": [[1,3,5],[1,3,5],[1,3,5]],
            "upsample_rates": [10, 10, 2, 2],
            "upsample_initial_channel": 512,
            "upsample_kernel_sizes": [16, 16, 4, 4],  # trebuie 16, nu 20
            "use_spectral_norm": False, "gin_channels": 256, "spk_embed_dim": 109
        }
    }
    with open(os.path.join(log_dir, 'config.json'), 'w') as f:
        json.dump(config, f, indent=2)


# ── Fix 2: matplotlib >= 3.8 a eliminat tostring_rgb ───────────────────────
def _patch_matplotlib(rvc_dir):
    path = os.path.join(rvc_dir, 'infer/lib/train/utils.py')
    if not os.path.exists(path):
        return
    code = open(path).read()
    if 'tostring_rgb' not in code:
        return
    old = 'np.fromstring(fig.canvas.tostring_rgb(), dtype=np.uint8, sep="")'
    new = ('np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)'
           '.reshape(fig.canvas.get_width_height()[1], fig.canvas.get_width_height()[0], 4)[:, :, :3]')
    open(path, 'w').write(code.replace(old, new))
    print('  patch matplotlib OK')


# ── Fix 3: HuBERT cu transformers (fairseq incompatibil cu Python 3.12) ────
def _extract_hubert(log_dir):
    from transformers import HubertModel, Wav2Vec2FeatureExtractor
    wav_dir = os.path.join(log_dir, '1_16k_wavs')
    out_dir = os.path.join(log_dir, '3_feature768')
    os.makedirs(out_dir, exist_ok=True)

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    is_half = device == 'cuda'
    print(f'  HuBERT pe {device}...')
    extractor = Wav2Vec2FeatureExtractor.from_pretrained('facebook/hubert-base-ls960')
    model = HubertModel.from_pretrained('facebook/hubert-base-ls960').to(device)
    if is_half:
        model = model.half()
    model.eval()

    wavs = sorted(f for f in os.listdir(wav_dir) if f.endswith('.wav'))
    print(f'  Procesare {len(wavs)} fisiere...')
    for i, fname in enumerate(wavs):
        out = os.path.join(out_dir, fname.replace('.wav', '.npy'))
        if os.path.exists(out):
            continue
        audio, _ = sf.read(os.path.join(wav_dir, fname))
        audio = audio.astype(np.float32)
        if audio.ndim > 1:
            audio = audio.mean(1)
        inp = extractor(audio, sampling_rate=16000, return_tensors='pt', padding=True)
        with torch.no_grad():
            vals = inp.input_values.to(device)
            if is_half:
                vals = vals.half()
            feats = model(vals, output_hidden_states=True).hidden_states[9][0].float().cpu().numpy()
        np.save(out, feats)
        if (i + 1) % 50 == 0 or i == len(wavs) - 1:
            print(f'  [{i+1}/{len(wavs)}]')
    print('  HuBERT gata.')


# ── Fix 4: filelist.txt cu ordinea corecta a coloanelor ────────────────────
# data_utils.py asteapta: wav | features(phone) | f0(pitch) | f0nsf | speaker_id
def _create_filelist(log_dir):
    wavs = sorted(glob.glob(f'{log_dir}/0_gt_wavs/*.wav'))
    lines, skipped = [], 0
    for wav in wavs:
        base = os.path.basename(wav)
        stem = base[:-4]
        feat   = f'{log_dir}/3_feature768/{stem}.npy'
        f0     = f'{log_dir}/2a_f0/{base}.npy'
        f0nsf  = f'{log_dir}/2b-f0nsf/{base}.npy'
        if not all(os.path.exists(p) for p in [feat, f0, f0nsf]):
            skipped += 1
            continue
        lines.append(f'{wav}|{feat}|{f0}|{f0nsf}|0')
    with open(f'{log_dir}/filelist.txt', 'w') as f:
        f.write('\n'.join(lines))
    print(f'  filelist.txt: {len(lines)} intrari ({skipped} sarite)')


# ── Watcher: copiaza checkpoints pe Drive la fiecare 60s ───────────────────
def _watch(log_dir, models_out, voice_id, stop):
    seen = set()
    while not stop.is_set():
        for p in glob.glob(f'{log_dir}/*.pth'):
            if p not in seen:
                try:
                    shutil.copy2(p, f'{models_out}/{voice_id}_ckpt_{os.path.basename(p)}')
                    seen.add(p)
                    print(f'  [Drive] {os.path.basename(p)} salvat')
                except Exception:
                    pass
        time.sleep(60)


# ── Pipeline complet pentru o voce ─────────────────────────────────────────
def train_voice(v):
    vid       = v['voice_id']
    sr        = v['sr']
    epochs    = v['epochs']
    batch     = v['batch']
    save_ev   = v['save_every']
    clips_dir = f"{CLIPS_BASE}/{vid}/clips"
    log_dir   = f'{RVC_DIR}/logs/{vid}'
    os.makedirs(log_dir, exist_ok=True)

    print(f'\n{"="*55}')
    print(f'  {v["name"]} ({vid}) — {epochs} epoci, batch {batch}')
    print(f'{"="*55}')

    _create_config(log_dir, sr, batch)
    _patch_matplotlib(RVC_DIR)

    stop = threading.Event()
    threading.Thread(target=_watch, args=(log_dir, MODELS_OUT, vid, stop), daemon=True).start()

    try:
        print('\n[1/4] Preprocessing...')
        subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/preprocess.py',
            clips_dir, str(sr), '2', log_dir, 'False', '3.7',
        ], check=True, cwd=RVC_DIR)

        print('\n[2/4] Extractare features HuBERT...')
        _extract_hubert(log_dir)

        print('\n[3/4] Extractare F0...')
        subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/extract/extract_f0_print.py',
            log_dir, '2', 'rmvpe',
        ], check=True, cwd=RVC_DIR)

        print('\n[3.5/4] Creare filelist...')
        _create_filelist(log_dir)

        print(f'\n[4/4] Antrenare {epochs} epoci...')
        r = subprocess.run([
            'python', f'{RVC_DIR}/infer/modules/train/train.py',
            '-e', vid, '-sr', f'{sr//1000}k', '-f0', '1',
            '-bs', str(batch), '-g', '0',
            '-te', str(epochs), '-se', str(save_ev),
            '-pg', f'{RVC_DIR}/assets/pretrained_v2/f0G{sr//1000}k.pth',
            '-pd', f'{RVC_DIR}/assets/pretrained_v2/f0D{sr//1000}k.pth',
            '-l', '0', '-c', '0', '-sw', '0', '-v', 'v2',
        ], stderr=subprocess.PIPE, text=True, cwd=RVC_DIR)

        if r.stderr:
            errs = [l for l in r.stderr.splitlines()
                    if any(k in l for k in ('Error', 'Traceback', 'INFO:voice'))]
            if errs:
                print('[train.py]', '\n'.join(errs[-20:]))
        if r.returncode != 0:
            raise RuntimeError(f'train.py cod {r.returncode}')

    finally:
        stop.set()

    ckpts = sorted(glob.glob(f'{log_dir}/*.pth'))
    if ckpts:
        dest = f'{MODELS_OUT}/{vid}.pth'
        shutil.copy2(ckpts[-1], dest)
        print(f'\n  Model salvat: {dest}')
    else:
        print('\n  EROARE: niciun checkpoint gasit.')


print('Functii definite. Continua cu celula urmatoare.')

In [ ]:
# Celula 8 — Functie construire index FAISS
import faiss, numpy as np

def build_index(vid):
    feat_dir = f'{RVC_DIR}/logs/{vid}/3_feature768'
    if not os.path.exists(feat_dir):
        print(f'Lipseste: {feat_dir}')
        return
    files = sorted(glob.glob(f'{feat_dir}/*.npy'))
    if not files:
        print('Nu exista features.')
        return
    feats = np.concatenate([np.load(f) for f in files], axis=0)
    print(f'  Features: {feats.shape}')
    n_ivf = min(int(16 * feats.shape[0]**0.5), feats.shape[0])
    idx = faiss.index_factory(768, f'IVF{n_ivf},Flat')
    idx.train(feats)
    idx.add(feats)
    idx_path = f'{RVC_DIR}/logs/{vid}/{vid}_v2.index'
    faiss.write_index(idx, idx_path)
    shutil.copy2(idx_path, f'{MODELS_OUT}/{vid}.index')
    print(f'  Index salvat: {MODELS_OUT}/{vid}.index')

print('Functia build_index definita.')

In [ ]:
# Celula 9 — ANTRENARE (ruleaza aceasta celula ultima)
# Dureaza ~1-2h per voce pe T4 GPU. Nu inchide Colab cat timp ruleaza.

for v in VOICES:
    try:
        train_voice(v)
        build_index(v['voice_id'])
        print(f"\n  DONE: {v['name']}")
    except Exception as e:
        import traceback
        print(f"\n  EROARE {v['voice_id']}: {e}")
        traceback.print_exc()

print('\n' + '='*55)
print('Antrenare completa. Modele salvate in Drive/disertatie/models/')
print('Urmatorul pas: 03_main_app.ipynb')
print('='*55)